# Hour 1 · Testing the “optimal alphabet” hypothesis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alexsosn/ugarit-dh-workshop/blob/main/notebooks/1b_alphabet_hypothesis.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/alexsosn/ugarit-dh-workshop/main?labpath=notebooks%2F1b_alphabet_hypothesis.ipynb)


**Goal:** test the popular claim (associated with **Jared Diamond**) that the Ugaritic alphabet is *optimally designed* — that **frequent letters are simpler** and/or come **earlier** in the alphabet.

We have the real ingredients: sign **frequencies** from the cuneiform corpus, and a **complexity** score (wedges + turns) for each of the 30 signs.

*Reading:* `docs/03-alphabet-and-language.md`.

## 0. Setup


In [ ]:
# === SETUP — run me first (works in Google Colab, Binder, and locally) ===
import os, sys, subprocess

if "google.colab" in sys.modules:                      # we're on Colab
    REPO_URL = "https://github.com/alexsosn/ugarit-dh-workshop.git"
    REPO_DIR = "/content/ugarit-dh-workshop"
    if not os.path.isdir(REPO_DIR):                    # clone the repo once
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(os.path.join(REPO_DIR, "notebooks"))      # work from notebooks/
    # Colab already ships numpy/pandas/scikit-learn/matplotlib/plotly/networkx;
    # only umap-learn is usually missing. Install just that (fast).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "umap-learn"], check=False)

# make data/loader.py importable (we run from the notebooks/ folder)
sys.path.insert(0, os.path.abspath(".."))
from data.loader import load_texts

texts = load_texts()     # 1st Colab run downloads the CUC JSONL from HuggingFace, then caches it
print(f"Loaded {len(texts)} tablets — genre of the first one: {texts[0]['genre']}")

## 1. The alphabet, with a complexity score
`complexity = wedges + turns` per sign, in canonical (abecedary) order.

In [ ]:
import pandas as pd
from data.loader import load_alphabet
alpha = pd.DataFrame(load_alphabet())
alpha.head(10)

## 2. Count how often each sign occurs
Counted exactly from the **cuneiform** text (one codepoint = one sign).

In [ ]:
from data.loader import sign_counts
counts = sign_counts(texts)
alpha["frequency"] = alpha["sign"].map(counts).fillna(0).astype(int)
alpha[["position","sign","char","complexity","frequency"]].head(10)

## 3. Frequency in alphabet order

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(11,4))
plt.bar(alpha["sign"], alpha["frequency"])
plt.xlabel("sign (abecedary order →)")
plt.ylabel("frequency"); plt.title("Ugaritic sign frequency in alphabet order"); plt.show()

## 4. Claim A — do *earlier* signs occur more often?

In [ ]:
cA = alpha["position"].corr(alpha["frequency"])
print(f"corr(position, frequency) = {cA:.3f}")
# strongly negative would support \"frequent letters come first\".

## ✍️ Your turn: Claim A with the South Semitic order

Ugaritic abecedaria are known in two alphabetic orders. The order used above is the Northern Semitic order, close to Phoenician, Hebrew, the earlier Arabic `ʾabjadī` order, and more distantly Greek and Latin. A second order, the South Semitic order, is closer to South Arabian and Geʽez.

**Beginner task:** edit Python code in the next cell. Replace every `TODO` with the right sign id from this South Semitic order:

`h l ḥ m q w š r t s k n ḫ b p ʾa ʿ ẓ g d ġ ṭ z ḏ y ṯ ṣ ʾi ʾu s2`

In the dataset, the three aleph signs are named `a`, `i`, and `u` instead of `ʾa`, `ʾi`, and `ʾu`. Keep the quotation marks and commas; only change the text inside quotes. Then run the cell and compare the two correlations.


In [ ]:
# Your turn: replace each "TODO" with the correct sign id.
# Keep the quotation marks and commas. Example: change "TODO" to "h".
# Dataset ids for aleph signs: ʾa = "a", ʾi = "i", ʾu = "u".
south_order = [
    "TODO", "l", "ḥ", "m", "TODO", "w", "š", "r", "t", "s",
    "k", "n", "TODO", "b", "p", "TODO", "ʿ", "ẓ", "g", "d",
    "ġ", "ṭ", "z", "TODO", "y", "ṯ", "ṣ", "TODO", "TODO", "s2",
]

# Validation + Claim A calculation. No edits needed below this line.
todo_positions = [pos for pos, sign in enumerate(south_order, start=1) if sign == "TODO"]
expected_signs = set(alpha["sign"])
given_signs = set(south_order) - {"TODO"}
duplicates = sorted({sign for sign in south_order if south_order.count(sign) > 1 and sign != "TODO"})
missing = sorted(expected_signs - given_signs)
extra = sorted(given_signs - expected_signs)

if todo_positions:
    print("First edit the TODO entries at positions:", todo_positions)
elif len(south_order) != 30 or missing or extra or duplicates:
    print("Check the list again.")
    print("length:", len(south_order), "(should be 30)")
    print("missing:", missing)
    print("extra:", extra)
    print("duplicates:", duplicates)
else:
    south_positions = {sign: pos for pos, sign in enumerate(south_order, start=1)}
    south = alpha.copy()
    south["south_position"] = south["sign"].map(south_positions)
    south = south.sort_values("south_position")

    cA_north = alpha["position"].corr(alpha["frequency"])
    cA_south = south["south_position"].corr(south["frequency"])
    print(f"Claim A, Northern/canonical order: corr(position, frequency) = {cA_north:.3f}")
    print(f"Claim A, South Semitic order:     corr(position, frequency) = {cA_south:.3f}")

    plt.figure(figsize=(11, 4))
    plt.bar(south["sign"], south["frequency"])
    plt.xlabel("sign (South Semitic order →)")
    plt.ylabel("frequency")
    plt.title("Ugaritic sign frequency in South Semitic order")
    plt.show()

    display(south[["south_position", "sign", "char", "frequency"]])


## 5. Claim B — are *frequent* signs *simpler*?

In [ ]:
cB = alpha["complexity"].corr(alpha["frequency"])
print(f"corr(complexity, frequency) = {cB:.3f}")

plt.figure(figsize=(7,5))
plt.scatter(alpha["complexity"], alpha["frequency"])
for _, r in alpha.iterrows():
    plt.annotate(r["sign"], (r["complexity"], r["frequency"]))
plt.xlabel("complexity (wedges + turns)"); plt.ylabel("frequency")
plt.title("Are frequent signs simpler?"); plt.show()

## 6. Discussion
- What do the two correlations actually say — is the “optimal design” claim supported, weak, or absent?
- **Caveats:** the corpus is damaged in places; complexity is one simple operationalisation; transliteration ≠ the scribe's hand.
- **Compare:** Christopher Wolfram ran a related analysis on **UDB** — a “two corpora, same question” check.

## ✍️ Your turn
Edit one value below and re-run. Nothing here can break the notebook — if it goes sideways, just re-run the setup cell.


In [ ]:
# Does the result change for one genre only? Try "myth", "letter", "ritual", "divination".
MY_GENRE = "myth"
subset = [t for t in texts if t["genre"] == MY_GENRE]
sc = sign_counts(subset)
alpha["freq_subset"] = alpha["sign"].map(sc).fillna(0).astype(int)
cA = alpha["position"].corr(alpha["freq_subset"])
cB = alpha["complexity"].corr(alpha["freq_subset"])
print(f"{MY_GENRE}: corr(position,freq)={cA:.3f}  corr(complexity,freq)={cB:.3f}")
# Hint: do the correlations get stronger or weaker on a single genre? Why might that be?